# Data Preprocessing
**Goal:** Extract features, construct ground truth (RUL labels), apply sliding windows, and prepare the final tensor/dataframe for LSTM training.
*   Step 1: Feature Extraction (max FFT bandwidth 1280 Hz)
*   Step 2: Construct Ground Truth (CUSUM)
*   Step 3: Sliding Window Generation & Merge

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import random
from typing import List, Dict, Tuple
from tqdm import tqdm

# Add local source to path for imports
sys.path.append("../src")
from CrossDomainFeatureExtractor import CrossDomainFeatureExtractor
from ConstructGroundTruthUsingCUSUM import UnivariateCUSUMDetector

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_PATH = r"../Processed_Data/Downsampled"
OUTPUT_PATH = r"../Processed_Data/LSTM_Inputs"
os.makedirs(OUTPUT_PATH, exist_ok=True)

IS_VAL_SET = False  # Toggle for train vs validation/test behavior. IS_VAL_SET = True forces saving separated windows to prevent timeline jumps
WINDOW_SIZES = [30, 40, 50, 70] # Generate multiple window sizes for both train and val/test

XJTU_SAMPLING_RATE = 25600.0
MAX_FREQ_HZ = 1280.0 # Domain constraint to match LAB Dataset

print(f"Input Path: {os.path.abspath(INPUT_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")
print(f"Is Val Set: {IS_VAL_SET}")
print(f"Window Sizes: {WINDOW_SIZES}")

In [ ]:
def create_sliding_windows(data_df: pd.DataFrame, window_size: int) -> pd.DataFrame:
    """Transforms continuous features into sliding window sequences (3D -> 2D flattened if needed)."""
    if len(data_df) <= window_size:
        return pd.DataFrame() # Return empty DataFrame if the sequence length is shorter than the window size

    windows = []
    features_col = [c for c in data_df.columns if c not in ["Change_Point", "RUL_Label", "Minute", "Bearing_ID", "Status"]]
    
    label_col = "RUL_Label"
    
    # Iterate through row sequence to make windows
    for i in range(len(data_df) - window_size):
        window_segment = data_df.iloc[i : i + window_size]
        
        # Flattened Window (e.g. td_rms_t0, td_rms_t1... )
        row_record = {}
        for feat in features_col:
            for t in range(window_size):
                row_record[f"{feat}_t{t}"] = window_segment.iloc[t][feat]
                
        # the label for prediction is typically the RUL of the LAST timestamp in window
        row_record['Target_RUL'] = window_segment.iloc[-1][label_col]
        row_record['Bearing_ID'] = window_segment.iloc[-1]['Bearing_ID']
        windows.append(row_record)
        
    return pd.DataFrame(windows)

def extract_features_and_build_gt(file_path: str, bearing_id: str) -> pd.DataFrame:
    """1. Extracts Features per minute. 2. Constructs GT using CUSUM."""
    # Use read_pickle as Notebook 1 outputs exact numpy structures as pickle objects
    df_raw = pd.read_pickle(file_path)
    
    extractor = CrossDomainFeatureExtractor(sampling_rate=XJTU_SAMPLING_RATE, max_freq_hz=MAX_FREQ_HZ)
    
    features_list = []
    
    for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc=f"Ext. {bearing_id}", leave=False):
        # Grab native Numpy array exactly as captured from raw csv
        try:
            h_sig = np.array(row['H_Signal'], dtype=float)
        except:
            h_sig = np.zeros(32768) # fallback if empty
            
        # Extract features from Horizontal signal (or vertical depending on setup)
        feats = extractor.extract_all_features(h_sig)
        feats['Minute'] = row['Minute']
        feats['Bearing_ID'] = bearing_id
        features_list.append(feats)
        
    df_features = pd.DataFrame(features_list)
    
    # Run CUSUM on RMS
    detector = UnivariateCUSUMDetector(baseline_ratio=0.1, k_factor=0.5, h_factor=5.0)
    change_point, _ = detector.fit_predict(df_features['td_rms'].values)
    
    # Calculate RUL Label
    # Healthy (before TCP): max RUL
    # Degradation (after TCP): linear decay
    total_life = len(df_features)
    rul_labels = []
    for min_idx in range(total_life):
        minute = min_idx + 1
        if minute <= change_point:
            # RUL remains constant at max degradation RUL
            rul_labels.append(total_life - change_point)
        else:
            rul_labels.append(total_life - minute)
            
    df_features['Change_Point'] = change_point
    df_features['RUL_Label'] = rul_labels
    return df_features

def preprocess_and_slide():
    # Load available files directly from INPUT_PATH
    if not os.path.exists(INPUT_PATH):
        print(f"Directory {INPUT_PATH} not found.")
        return
        
    all_bearings_features = {}
    
    # Search for exactly generated pickle files
    downsampled_files = [f for f in os.listdir(INPUT_PATH) if f.endswith(".pkl")]
    if not downsampled_files:
        print(f"No Pickled files found in {INPUT_PATH}. Looking iteratively inside condition folders...")
        for cond_folder in os.listdir(INPUT_PATH):
            cf = os.path.join(INPUT_PATH, cond_folder)
            if os.path.isdir(cf):
                downsampled_files.extend([os.path.join(cond_folder, f) for f in os.listdir(cf) if f.endswith(".pkl")])
                
    if not downsampled_files:
        print("No Pickle Files found. Make sure to run Notebook 1 first!")
        return
    
    # Step 1 & 2: Feature Extraction and Ground Truth
    print("--- Step 1 & 2: Feature Extraction & Ground Truth Construction ---")
    for file in tqdm(downsampled_files, desc=f"Bearings in {os.path.basename(INPUT_PATH)}"):
        # file might look like Condition_1/Bearing1_1_downsampled.pkl or Bearing1_1_downsampled.pkl
        bearing_id = os.path.basename(file).replace("_downsampled.pkl", "")
        file_path = os.path.join(INPUT_PATH, file)
        
        df_feat = extract_features_and_build_gt(file_path, bearing_id)
        all_bearings_features[bearing_id] = df_feat

    # Step 3: Sliding Windows
    print("\n--- Step 3: Sliding Windows & Output Compilation ---")
    
    for w_size in WINDOW_SIZES:
        if not IS_VAL_SET:
            # Training set logic: MERGED and SHUFFLED cross-bearing
            combined_w = []
            for b_id, df_feat in all_bearings_features.items():
                w_df = create_sliding_windows(df_feat, w_size)
                if not w_df.empty:
                    combined_w.append(w_df)
                else:
                    print(f"Warning: Skipped {b_id} for window size {w_size} (Insufficient data length: {len(df_feat)})")
                
            if combined_w:
                final_train_w = pd.concat(combined_w, ignore_index=True)
                # Overfitting prevention layout: SHUFFLE
                final_train_w = final_train_w.sample(frac=1, random_state=42).reset_index(drop=True)
                
                out_file = os.path.join(OUTPUT_PATH, f"processed_train_w{w_size}.csv")
                final_train_w.to_csv(out_file, index=False)
                print(f"Saved (Merged Train): {out_file} (Shape: {final_train_w.shape})")
            else:
                print(f"Error: No valid training data produced for window size {w_size}.")
        else:
            # Validation/Test set logic: Separated CSVs, NO SHUFFLING
            # This logic avoids cross-bearing chronological jumping on test metrics and RUL curves
            for b_id, df_feat in all_bearings_features.items():
                w_df = create_sliding_windows(df_feat, w_size)
                
                if w_df.empty:
                    print(f"Warning: Skipped {b_id} for window size {w_size} (Insufficient data length: {len(df_feat)})")
                    continue
                    
                out_file = os.path.join(OUTPUT_PATH, f"processed_val_{b_id}_w{w_size}.csv")
                w_df.to_csv(out_file, index=False)
                print(f"Saved (Isolated Val): {out_file} (Shape: {w_df.shape})")

# Execute Pipeline
preprocess_and_slide()